In [ ]:
from datetime import UTC, datetime, timedelta
import json
import pathlib
import zoneinfo

import numpy as np
import pandas as pd
import plotly.express as px
from skyfield import almanac
from skyfield.api import load, load_file, wgs84

In [ ]:
info_file = pathlib.Path("~/Documents/location_info.json").expanduser()
with info_file.open() as ifile:
    info = json.load(ifile)
latitude = info["latitude"]
longitude = info["longitude"]
location_timezone = "America/Phoenix"
location = wgs84.latlon(latitude, longitude)
zone = zoneinfo.ZoneInfo(location_timezone)

In [ ]:
ts = load.timescale()
eph = load_file("~/skyfield/de421.bsp")

In [ ]:
now = datetime.now(zone)
midnight = now.replace(hour=0, minute=0, second=0, microsecond=0)
next_midnight = midnight + timedelta(days=30)

In [ ]:
t0 = ts.from_datetime(midnight)
t1 = ts.from_datetime(next_midnight)

In [ ]:
f = almanac.dark_twilight_day(eph, location)
times, events = almanac.find_discrete(t0, t1, f)

In [ ]:
previous_e = f(t0).item()

In [ ]:
sunrise_list = []
sunset_list = []
for t, e in zip(times, events, strict=False):
    if previous_e < e:
        key = f"{almanac.TWILIGHTS[e]} starts"
        if "Day starts" in key:
            sunrise_list.append(t.astimezone(zone))
    else:
        key = f"{almanac.TWILIGHTS[previous_e]} ends"
        if "Day ends" in key:
            sunset_list.append(t.astimezone(zone))
    previous_e = e

In [ ]:
sunrises = np.array(sunrise_list)
sunsets = np.array(sunset_list)

In [ ]:
tod = np.array([(x.total_seconds() / 3600) for x in (sunsets - sunrises)])

In [ ]:
tod

In [ ]:
day = np.array([x.date() for x in sunrises])

In [ ]:
day

In [ ]:
df = pd.DataFrame(dict(date=day, tod=tod))
fig = px.line(df, x="date", y="tod")
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Length of Day (hours)")
fig.show()

In [ ]:
def time_decimal(t):
    d = t.hour + t.minute / 60. + t.second / 3600.
    return d

df_sr = pd.DataFrame(dict(date=day, sunrise=np.array([time_decimal(x.time()) for x in sunrises])))
df_ss = pd.DataFrame(dict(date=day, sunset=np.array([time_decimal(x.time()) for x in sunsets])))
fig = px.line(df_sr, x="date", y="sunrise")
fig.add_trace(px.line(df_ss, x="date", y="sunset").data[0])
fig.update_xaxes(title_text="Date")
fig.update_yaxes(title_text="Time of Day (hours)")
fig.update_layout(yaxis_range=[0,24])
fig.show()